In [1]:
import pandas as pd

df = pd.read_csv('products.csv')
df

,product_id,product_name,ingredient_group,cost_per_100g_eur,energy_kcal_per_100g,volume_ml_per_100g,dietary_class,allergen_gluten,allergen_nuts,allergen_dairy,is_available
0,1,Carrots,vegetables,0.17,37.4,NaN,vegan,0,0,0,1
1,2,Organic Carrots,vegetables,0.47,43.3,NaN,vegan,0,0,0,1
2,3,Broccoli,vegetables,0.11,37.7,NaN,vegan,0,0,0,1
3,4,Organic Broccoli,vegetables,0.29,33.4,NaN,vegan,0,0,0,1
4,5,Cauliflower,vegetables,0.51,29.5,NaN,vegan,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...
3132,3133,Unsweetened Chocolate Oat Milk,plant-based beverages,0.36,64.1,97.7,vegan,0,0,0,1
3133,3134,Barista Chocolate Oat Milk,plant-based beverages,0.14,83.2,101.3,vegan,0,0,0,1
3134,3135,Vanilla Almond Milk,plant-based beverages,0.26,41.1,108.3,vegan,0,1,0,1
3135,3136,Unsweetened Vanilla Almond Milk,plant-based beverages,0.14,36.9,102.0,vegan,0,1,0,1


## 1. Data Quality Checks

In [2]:
print("=== Shape ===")
print(f"{df.shape[0]} rows, {df.shape[1]} columns\n")

print("=== Dtypes ===")
print(df.dtypes, "\n")

print("=== Null counts ===")
nulls = df.isnull().sum()
# volume_ml_per_100g being null for non-liquids is documented/expected
print(nulls[nulls > 0], "\n")

print("=== Duplicates ===")
print(f"Duplicate product_id:   {df['product_id'].duplicated().sum()}")
print(f"Duplicate product_name: {df['product_name'].duplicated().sum()}\n")

print("=== dietary_class distribution ===")
print(df['dietary_class'].value_counts(), "\n")

print("=== is_available distribution ===")
print(df['is_available'].value_counts(), "\n")

print("=== ingredient_group distribution (top 10) ===")
print(df['ingredient_group'].value_counts().head(10), "\n")

print("=== Allergen binary check (expect only 0 / 1) ===")
for col in ['allergen_gluten', 'allergen_nuts', 'allergen_dairy', 'is_available']:
    unexpected = df[~df[col].isin([0, 1])][col].unique()
    print(f"  {col}: unexpected values → {unexpected if len(unexpected) else 'none'}")

=== Shape ===
3137 rows, 11 columns

=== Dtypes ===
product_id                int64
product_name             object
ingredient_group         object
cost_per_100g_eur       float64
energy_kcal_per_100g    float64
volume_ml_per_100g      float64
dietary_class            object
allergen_gluten           int64
allergen_nuts             int64
allergen_dairy            int64
is_available              int64
dtype: object 

=== Null counts ===
volume_ml_per_100g    2640
dtype: int64 

=== Duplicates ===
Duplicate product_id:   0
Duplicate product_name: 0

=== dietary_class distribution ===
dietary_class
vegan         1843
vegetarian     684
meat           610
Name: count, dtype: int64 

=== is_available distribution ===
is_available
1    2959
0     178
Name: count, dtype: int64 

=== ingredient_group distribution (top 10) ===
ingredient_group
cheeses       378
vegetables    202
spices        200
fruits        189
meat          171
sauces        136
condiments    120
pastas        118
fish     

## 2. Filter: vegan / vegetarian + available only

In [3]:
df_filtered = df[
    df['dietary_class'].isin(['vegan', 'vegetarian']) &
    (df['is_available'] == 1)
].copy()

print(f"Rows before filtering : {len(df)}")
print(f"Rows after filtering  : {len(df_filtered)}")
print(f"Dropped               : {len(df) - len(df_filtered)}\n")
print("dietary_class breakdown in filtered set:")
print(df_filtered['dietary_class'].value_counts())
df_filtered.head()

Rows before filtering : 3137
Rows after filtering  : 2382
Dropped               : 755

dietary_class breakdown in filtered set:
dietary_class
vegan         1738
vegetarian     644
Name: count, dtype: int64


,product_id,product_name,ingredient_group,cost_per_100g_eur,energy_kcal_per_100g,volume_ml_per_100g,dietary_class,allergen_gluten,allergen_nuts,allergen_dairy,is_available
0,1,Carrots,vegetables,0.17,37.4,NaN,vegan,0,0,0,1
1,2,Organic Carrots,vegetables,0.47,43.3,NaN,vegan,0,0,0,1
2,3,Broccoli,vegetables,0.11,37.7,NaN,vegan,0,0,0,1
3,4,Organic Broccoli,vegetables,0.29,33.4,NaN,vegan,0,0,0,1
4,5,Cauliflower,vegetables,0.51,29.5,NaN,vegan,0,0,0,1


## 3. Meal component grouping

Map each `ingredient_group` to one of four meal-component roles that define a balanced main:

| Role | ingredient_groups |
|---|---|
| `carb_base` | grains, rices, pastas, breads |
| `protein` | legumes, eggs, plant proteins, nuts, seeds, cheeses |
| `vegetables` | vegetables, mushrooms, fruits |
| `sauce_dressing` | sauces, condiments, oils, creams and butters, yogurts |
| `other` | flours, milks, plant-based beverages, sweeteners, spices, herbs |

In [4]:
MEAL_COMPONENT_MAP = {
    # Carb base
    'grains':               'carb_base',
    'rices':                'carb_base',
    'pastas':               'carb_base',
    'breads':               'carb_base',
    # Protein
    'legumes':              'protein',
    'eggs':                 'protein',
    'plant proteins':       'protein',
    'nuts':                 'protein',
    'seeds':                'protein',
    'cheeses':              'protein',   # dairy protein counts for vegetarian track
    # Vegetables
    'vegetables':           'vegetables',
    'mushrooms':            'vegetables',
    'fruits':               'vegetables',
    # Sauce / dressing
    'sauces':               'sauce_dressing',
    'condiments':           'sauce_dressing',
    'oils':                 'sauce_dressing',
    'creams and butters':   'sauce_dressing',
    'yogurts':              'sauce_dressing',
    # Supporting / background ingredients — not a primary meal component
    'flours':               'other',
    'milks':                'other',
    'plant-based beverages':'other',
    'sweeteners':           'other',
    'spices':               'other',
    'herbs':                'other',
}

df_filtered['meal_component'] = df_filtered['ingredient_group'].map(MEAL_COMPONENT_MAP)

# Sanity-check: any ingredient_group not covered by the map?
unmapped = df_filtered[df_filtered['meal_component'].isna()]['ingredient_group'].unique()
if len(unmapped):
    print(f"Unmapped ingredient groups: {unmapped}")
else:
    print("All ingredient groups mapped successfully\n")

print("meal_component distribution:")
print(df_filtered['meal_component'].value_counts())
df_filtered[['product_name', 'ingredient_group', 'meal_component']].head(10)

All ingredient groups mapped successfully

meal_component distribution:
meal_component
protein           618
other             530
vegetables        459
sauce_dressing    427
carb_base         348
Name: count, dtype: int64


,product_name,ingredient_group,meal_component
0,Carrots,vegetables,vegetables
1,Organic Carrots,vegetables,vegetables
2,Broccoli,vegetables,vegetables
3,Organic Broccoli,vegetables,vegetables
4,Cauliflower,vegetables,vegetables
5,Organic Cauliflower,vegetables,vegetables
6,Spinach,vegetables,vegetables
7,Organic Spinach,vegetables,vegetables
8,Kale,vegetables,vegetables
9,Organic Kale,vegetables,vegetables
